# Data Preprocessing

# Loading the Building-energy-dataset

In [ ]:
import numpy as np
import pandas as pd
import glob

path = '../input/building-energy-dataset'   # Directory path containing CSV files
 # use your path
all_files = glob.glob(path + "/*.csv")      # Find all CSV files in the directory    # https://docs.python.org/3/library/glob.html

li = []
for filename in all_files:
    df = pd.read_csv(filename, index_col="Time",parse_dates=True, header=0)          # https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html
    li.append(df)

building = pd.concat(li, axis=0, ignore_index=False)
building.sort_index(inplace= True)

building.info()

The path is where the CSV files are located. 

glob.glob is used to find all CSV files in the directory. https://docs.python.org/3/library/glob.html

pd.read_csv reads each CSV file into a DataFrame, using "Time" as the index and parsing dates. Reference: 

pandas.read_csv   https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html


Each DataFrame is added to a list li.



In [ ]:

building.columns = ['HVAC Ele. [kW]', 'Chiller Ele. Power  [kW]', 'Humidifier Ele. power [kW]', 'light Power [kW]',' power[kW]','PV panels power [kW]','Battery system power']

building.head(5)



The data has 1-minute resolution and the values are given in power. We can resample the data to hourly resolution to perform initial analysis.

In [ ]:
# Convert the current index to a DatetimeIndex with day-first format
building.index = pd.to_datetime(building.index, dayfirst=True)

# Resample the data from 1-minute resolution to hourly
building = building.resample('h').sum()

# Display the first few rows
building.head()



# Loading the weather data

In [ ]:
path = '../input/weather-data/Weather_data.csv'
 # use your path
weather_data = pd.read_csv(path, index_col="Datetime",parse_dates=True, header=0)
column_names = {'GHI':'Global Horizontal Irradiance [W/m2]', 'DIF':'Diffuse Horizontal Irradiance [W/m2]', 'DNI':'Direct Normal Irradiance [W/m2]', 'SE':'Sun elevation angle [°]', 'SA':'Sun azimuth angle [°]', 'TMOD':'Module temperature [°C]', 
                'TEMP':'Air temperature [°C]', 'WS':'Wind speed [m/s]', 'WD':'Wind direction [°]', 'RH':'Relative humidity [%]', 'AP':' Atmospheric pressure [hPa]', 'PWAT':'Precipitable Water [kg/m2]', 'SWE':'Snow water equivalent [kg/m2]', 'WG':'Wind gust [m/s]'}
weather_data.rename(columns=column_names, inplace=True)
weather_data.head(5)

pd.read_csv reads the weather data CSV file into a DataFrame.    https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html


index_col="Datetime" sets the "Datetime" column as the index of the DataFrame.


parse_dates=True automatically converts the "Datetime" column to a datetime format, which is helpful for time-series analysis.


Resampling the weather data from 15 minute resolution to hourly to match with the energy consumption

In [ ]:
weather_data = weather_data.resample('h').mean()
weather_data.head()

Here we concatenate both frames together

In [ ]:
#concatenating the input and output parameters
df = pd.concat([weather_data, building], axis=1)
df.head()

The code snippet provided concatenates two DataFrames (weather_data and building) into a single DataFrame, df, by aligning them along their common index (time). Here’s a breakdown of what the code does:
pd.concat is a function from the pandas library that combines multiple DataFrames along a specified axis.

weather_data and building are two DataFrames that likely share the same time-based index (Datetime for weather_data and Time or similar for building).

axis=1 means the concatenation is done column-wise. It merges the two DataFrames side-by-side, aligning rows by their index (time). If any row in one DataFrame does not have a matching index in the other, the result will have NaN values for the missing data.
https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.concat.html

# Exploratory data analysis

In [ ]:
##### Plot the yearly actual loads
import matplotlib.pyplot as plt
#group data by year
groups = building['HVAC Ele. [kW]'].groupby(pd.Grouper(freq='YE'))

#set figure and axis
fig, axs = plt.subplots(len(groups), 1, figsize=(15,15))


for ax, (name, group) in zip(axs, groups):
    
    #plot the data
    ax.plot(pd.Series(group.values))

    ax.set_xlabel('Hour of Year')
    ax.set_ylabel('Total Load')
    ax.set_title(name.year)
    plt.subplots_adjust(hspace=0.5)
    

In [ ]:
# # Group data by year
# groups = building['HVAC Ele. [kW]'].groupby(pd.Grouper(freq='Y'))

# # Set figure and axis
# fig, axs = plt.subplots(len(groups), 1, figsize=(15, 15))

# # Ensure axs is always a 1D array of axes
# if len(groups) == 1:
#     axs = [axs]  # Make it iterable if there is only one group

# for ax, (name, group) in zip(axs, groups):
#     # Plot the data using day of year for better x-axis interpretation
#     ax.plot(group.index.dayofyear, group.values) 
#     ax.set_xlabel('Day of Year')
#     ax.set_ylabel('Total Load')
#     ax.set_title(name.year)

# plt.subplots_adjust(hspace=0.5)  # Adjust space between plots
# plt.show()


### Correlation heatmap

Let us follow the instructions of https://www.python-graph-gallery.com/91-customize-seaborn-heatmap

In [ ]:
corr = df.corr()   #This line calculates the correlation matrix for all numerical columns in the DataFrame df. The result is a new DataFrame, corr, where each element represents the correlation coefficient between two variables. 
corr.style.background_gradient(cmap='coolwarm')   #This line uses the style accessor of the DataFrame to apply a background gradient color mapping (cmap) to the correlation matrix. The coolwarm colormap will display negative correlations in cool colors (blue) and positive correlations in warm colors (red).

search for ***style.background_gradient method***

**Understanding the Correlation Coefficients**

The gradient makes it easier to quickly identify strong positive or negative correlations. It visually highlights patterns and relationships between variables in the data. 



The correlation coefficient values range from -1 to 1:


+1 indicates a perfect positive correlation, meaning as one variable increases, the other also increases.

-1 indicates a perfect negative correlation, meaning as one variable increases, the other decreases.

0 indicates no linear correlation between the variables.


Global Horizontal Irradiance [W/m2]:


Positively correlates with Direct Normal Irradiance (0.85) and Diffuse Horizontal Irradiance (0.84), which makes sense as both are components of global horizontal irradiance.

Moderately correlates with Sun elevation angle (0.77), indicating that as the sun elevation angle increases, so does the global horizontal irradiance.

Negatively correlates with PV panels power [kW] (-0.68), which might indicate that as more sunlight (irradiance) is available, less power is being drawn from the PV panels (possibly due to charging or energy storage dynamics).


In [ ]:
# If you want to save or display the correlation matrix as an image, you can use Seaborn and Matplotlib:

import seaborn as sns
import matplotlib.pyplot as plt

# Set the figure size
plt.figure(figsize=(10, 8))

# Create the heatmap
sns.heatmap(corr, annot=True, cmap='coolwarm', linewidths=0.5)

# Show the plot
plt.show()


sns.heatmap: Creates a heatmap for visualizing the correlation matrix.

corr: The DataFrame containing the correlation matrix.

annot=True: Annotates each cell with the numeric value of the correlation coefficient.

https://seaborn.pydata.org/generated/seaborn.heatmap.html

In [ ]:
# libraries
import seaborn as sns

# plot a heatmap with annotation
corr = df.corr()
sns.heatmap(corr, annot=True, annot_kws={"size": 7})

Following with hierarchical clustering
https://www.python-graph-gallery.com/405-dendrogram-with-heatmap-and-coloured-leaves

In [ ]:
# Libraries
import seaborn as sns
from matplotlib import pyplot as plt
 
# plot
sns.clustermap(corr, metric="correlation", method="single", cmap="Blues")  # sns.clustermap: This function creates a heatmap where rows and columns are clustered together based on similarity, making it easier to identify patterns or groupings in the data.
plt.show()

sns.clustermap: Creates a heatmap that clusters both rows and columns based on their similarity, using hierarchical clustering.

**Hierarchical clustering** is a method of clustering that builds a hierarchy of clusters by either:

Hierarchical clustering is a versatile and intuitive method for clustering data into groups based on their similarity, represented by a dendrogram. It is particularly useful when you don't know the number of clusters beforehand and want to explore the hierarchical structure of your data.

In the context of a hierarchical clustering heatmap, the terms "plus" (+) and "minus" (-) typically refer to the positive and negative correlation values between different variables or data points that are being clustered and visualized.

### Heatmap for time series

https://www.python-graph-gallery.com/heatmap-for-timeseries-matplotlib

In [ ]:
# Subset data
subset = building[(building.index.year == 2019) & (building.index.month == 6)]

# Extract hour, day, and temperature
hour = subset.index.hour
day = subset.index.day
data = subset['HVAC Ele. [kW]']

# Re-arrange temperature values
data = data.values.reshape(24, len(day.unique()), order="F")   #  https://numpy.org/doc/stable/reference/generated/numpy.reshape.html

# Compute x and y grids, passed to `ax.pcolormesh()`.

# The first + 1 increases the length
# The outer + 1 ensures days start at 1, and not at 0.
xgrid = np.arange(day.max() + 1) + 1

# Hours start at 0, length 2
ygrid = np.arange(25)

The reshape method is used to adjust the structure of an array, and the order="F" ensures that the data is filled column-by-column. This is particularly useful in time-series analysis and visualization scenarios, like plotting hourly data over days.



This code prepares the data for visualization, such as plotting a heatmap of HVAC electricity usage over each hour and day of June 2019. The xgrid and ygrid are created for use with a plotting function like pcolormesh in Matplotlib, which requires grid coordinates to display the data correctly.

xgrid: Creates an array of day numbers from 1 to max(day) + 1 for plotting along the x-axis. The addition ensures the days start at 1 rather than 0.

ygrid: Creates an array of hours from 0 to 24 for plotting along the y-axis. This array represents the hours of the day, including an extra point for the edge of the last hour.

Converts the HVAC electricity usage data into a 2D array of shape (24, number_of_days_in_June) using Fortran-style (column-major) ordering.

This reshaping is useful for creating a heatmap where rows represent hours (0–23), and columns represent days (1–30 or 1–31 depending on the month).

https://numpy.org/doc/stable/reference/generated/numpy.reshape.html

In [ ]:
fig, ax = plt.subplots()
ax.pcolormesh(xgrid, ygrid, data)
ax.set_frame_on(False) # remove all spines

a more complex heatmap plot...

In [ ]:
MIN_TEMP = building["HVAC Ele. [kW]"].min()  # Finds the minimum (MIN_TEMP) and maximum (MAX_TEMP) values of the "HVAC Ele. [kW]" column in the building DataFrame to set the color scale limits for the plot.
MAX_TEMP = building["HVAC Ele. [kW]"].max()

def single_plot(data, month, year, ax):   # data: The DataFrame to be filtered and plotted. month,# year: Time period to subset the data. #ax: The matplotlib axis object where the plot will be drawn.
    data = data[(data.index.year == year) & (data.index.month == month)]  #Selects the data for the specified month and year.

    hour = data.index.hour  # Extracts hour and day, reshapes the data into a 24-hour by number of days format for a heatmap.
    day = data.index.day
    temp = data.values.reshape(24, len(day.unique()), order="F")
    
    xgrid = np.arange(day.max() + 1) + 1   #Defines x and y grids for the plot.
    ygrid = np.arange(25)
    
    ax.pcolormesh(xgrid, ygrid, temp, cmap="magma", vmin=MIN_TEMP, vmax=MAX_TEMP)   #Plots the heatmap using a "magma" colormap, scaled between the minimum and maximum HVAC values.
    # Invert the vertical axis
    ax.set_ylim(24, 0)
    # Set tick positions for both axes
    ax.yaxis.set_ticks([i for i in range(24)])
    ax.xaxis.set_ticks([10, 20, 30])
    # Remove ticks by setting their length to 0
    ax.yaxis.set_tick_params(length=0)
    ax.xaxis.set_tick_params(length=0)
    
    # Remove all spines
    ax.set_frame_on(False)

In [ ]:
number_of_years_to_plot = building.index.year.max() - building.index.year.min()

fig, axes = plt.subplots(number_of_years_to_plot, 12, figsize=(30, 20), sharey=True)

for i, year in enumerate(range(building.index.year.min(), building.index.year.max())):
    for j, month in enumerate(range(1, 13)):
        single_plot(building["HVAC Ele. [kW]"], month, year, axes[i, j])

# Adjust margin and space between subplots
# Extra space is on the left to add a label
fig.subplots_adjust(left=0.05, right=0.98, top=0.9, hspace=0.08, wspace=0.04)